In [12]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [13]:
def get_db_context():

    return [
        {
            "content": "Enterprise customers generate 70% of total revenue.",
            "source": "database",
            "priority": 5,
            "type": "structured"
        }
    ]

In [14]:
def get_document_context():

    return [
        {
            "content": "Subscription pricing works best for enterprise SaaS.",
            "source": "documents",
            "priority": 3,
            "type": "unstructured"
        },
        {
            "content": "Freemium models often fail in enterprise environments.",
            "source": "documents",
            "priority": 4,
            "type": "unstructured"
        }
    ]

In [15]:
def get_api_context():

    return [
        {
            "content": "Competitor recently launched usage-based pricing.",
            "source": "api",
            "priority": 4,
            "type": "real-time"
        }
    ]

In [16]:
def collect_all_context():

    context = []
    context.extend(get_db_context())
    context.extend(get_document_context())
    context.extend(get_api_context())

    return context

In [17]:
def compute_relevance_score(text, query):

    # Simple keyword match (replace with embeddings if needed)
    score = 0

    for word in query.lower().split():
        if word in text.lower():
            score += 1

    return score

In [18]:
def rank_context(contexts, query):

    scored = []

    for c in contexts:

        relevance = compute_relevance_score(c["content"], query)

        base_priority = c.get("priority", 1)

        # Source weighting
        if c["source"] == "api":
            source_weight = 1.2
        elif c["source"] == "database":
            source_weight = 1.0
        else:
            source_weight = 0.8

        final_score = (relevance * 2 + base_priority) * source_weight

        scored.append((c, final_score))

    # Sort by score
    scored.sort(key=lambda x: x[1], reverse=True)

    return [c for c, _ in scored]

In [19]:
def select_top_context(contexts, k=3):

    return contexts[:k]

In [20]:
def build_context(contexts):

    formatted = []

    for c in contexts:
        formatted.append(f"[{c['source'].upper()}] {c['content']}")

    return "\n\n".join(formatted)

In [21]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

def generate_answer(query, context):

    llm = ChatOpenAI(model="gpt-4o", temperature=0, openai_api_key=os.getenv("OPENAI_API_KEY"))

    messages = [
        SystemMessage(content="You are a senior product strategist."),
        HumanMessage(content=f"""
Use the context below to answer the question.

Context:
{context}

Question:
{query}
""")
    ]

    return llm.invoke(messages).content

In [22]:
def multi_source_pipeline(query):

    # Step 1: Collect
    contexts = collect_all_context()

    # Step 2: Rank
    ranked = rank_context(contexts, query)

    # Step 3: Select
    top_context = select_top_context(ranked)

    # Step 4: Transform
    context_text = build_context(top_context)

    # Step 5: LLM
    answer = generate_answer(query, context_text)

    return answer

In [23]:
query = "What pricing strategy should we use for enterprise customers?"

response = multi_source_pipeline(query)

print(response)

Given the context, it would be advisable to continue using subscription pricing for enterprise customers. The document indicates that subscription pricing works best for enterprise SaaS, and the database shows that enterprise customers generate 70% of total revenue, suggesting that the current model is effective. While a competitor has launched usage-based pricing, it may not be necessary to shift strategies if the existing subscription model is already successful and well-received by your enterprise clients. However, it could be beneficial to monitor the competitor's performance with usage-based pricing to assess if any adjustments are needed in the future.
